In [12]:
import pandas as pd
from pydantic import BaseModel, validator
from typing import List, Set

# Step 1: Read the CSV file using pandas

In [13]:
SETS_Fuel_df = pd.read_csv('workflow/data/FUEL.csv')

In [19]:
SETS_Fuel_df

,VALUE
0,INDDSL
1,DSL
2,INDBIO
3,BIO
4,INDNGS
...,...
117,LFORBC1
118,LGRSBC1
119,LOTHBC1
120,LWATBC1


# Step 2: Define a Pydantic model

In [40]:
class BCNexus_inputs(BaseModel):
    FUEL: str
    COMMODITY: str
    EMISSION: str
    TECHNOLOGY: str
    InputActivityRatio : float


    @validator('FUEL','COMMODITY')
    def check_non_empty(cls, v):
        if not v.strip():
            raise ValueError('Fuel name cannot be empty')
        return v

/tmp/ipykernel_4098675/3371086567.py:4: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.6/migration/
  @validator('VALUE')


PydanticUserError: Decorators defined with incorrect fields: __main__.BCNexus_inputs:93865304841152.check_non_empty (use check_fields=False if you're inheriting from the model and intended this)

For further information visit https://errors.pydantic.dev/2.6/u/decorator-missing-field

# Step 3: Validate and import data as a set

In [38]:
def load_fuel_set(csv_file: str) -> Set[str]:
    df = pd.read_csv(csv_file)
    fuels = set()
    
    for record in df.to_dict(orient='records'):
        # Create a Pydantic model instance to validate data
        fuel = BCNexus_inputs(**record)
        fuels.add(fuel.VALUE)
    
    return fuels


In [39]:
# Usage
fuel_set = load_fuel_set('workflow/data/FUEL.csv')
# Convert the set to a DataFrame
fuel_df = pd.DataFrame(list(fuel_set), columns=['VALUE'])

# Save the DataFrame to a CSV file
fuel_df.to_csv('workflow/pydantic_test/FUEL.csv', index=False)